# Chapter 5: Agents and Tool Use

*Small Language Models in Practice — Haji Gul*

> Turning a passive model into something that *acts*; the tool-calling loop in
its simplest honest form; and a working agent that decides when to call a
calculator and a search function — built from scratch so you see every moving
part.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

; the tool-calling loop in
its simplest honest form; and a working agent that decides when to call a
calculator and a search function — built from scratch so you see every moving
part.

## What an agent really is

Strip away the hype and an **agent** is a loop: the model is told which
tools exist, it responds with a request to call one, your code runs that tool and
feeds the result back, and the loop repeats until the model produces a final
answer. The model never runs anything itself — *you* do, on its
instruction. That separation is the whole safety and control story.

> **The agent loop.** 1. Describe the tools to the model.;
2. Model emits a tool call (name + arguments).;
3. Your code executes the tool.;
4. Feed the result back.;
5. Repeat until the model answers instead of calling a tool.

## Step 1: define some tools

A tool is just a Python function plus a description the model can read.

In [ ]:
import json

def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: invalid characters"
    return str(eval(expression))   # safe: input is character-restricted

def word_count(text: str) -> str:
    """Count the words in a string."""
    return str(len(text.split()))

TOOLS = {"calculator": calculator, "word_count": word_count}

TOOL_SPEC = """
You can call tools by replying with ONLY a JSON object:
{"tool": "<name>", "args": {"<arg>": "<value>"}}
Available tools:
- calculator(expression): evaluate arithmetic, e.g. "12 * (3+4)"
- word_count(text): count words in text
When you have the final answer, reply with plain text (no JSON).
"""

## Step 2: the tool-calling loop

We parse the model's reply: if it is JSON, run the tool and continue; otherwise
it is the final answer.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation",
               model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto")

def run_agent(question, max_steps=4):
    messages = [
        {"role": "system", "content": TOOL_SPEC},
        {"role": "user", "content": question},
    ]
    for _ in range(max_steps):
        out = gen(messages, max_new_tokens=120, do_sample=False)
        reply = out[0]["generated_text"][-1]["content"].strip()

        try:
            call = json.loads(reply)            # did it ask for a tool?
        except json.JSONDecodeError:
            return reply                        # no -> final answer

        tool = TOOLS.get(call["tool"])
        result = tool(**call["args"]) if tool else "Error: unknown tool"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user",
                         "content": f"Tool result: {result}"})
    return "Stopped: too many steps."

## Step 3: watch it work

In [ ]:
print(run_agent("What is 1234 * 17, and how many words are in this question?"))

The model first emits a `calculator` call, gets the product, then a
`word_count` call, then writes the final sentence combining both —
exactly the loop from the start of the chapter.

> **Tip.** Smaller models follow strict output formats less reliably. Two cheap fixes:
keep the tool list short, and give one worked example in the system prompt. If a
model still drifts, step up one size (here 1.5B over 0.5B) before adding
frameworks.

## Using a framework

Hand-rolling teaches the mechanics; in production a small framework handles the
parsing, retries, and tracing. The loop is identical — only the boilerplate
disappears.

In [ ]:
%%bash
pip install smolagents

In [ ]:
from smolagents import CodeAgent, tool, TransformersModel

@tool
def calculator(expression: str) -> str:
    """Evaluate arithmetic. Args: expression: the expression to evaluate."""
    return str(eval(expression))

agent = CodeAgent(
    tools=[calculator],
    model=TransformersModel(model_id="Qwen/Qwen2.5-1.5B-Instruct"),
)
print(agent.run("What is 1234 * 17?"))

## Recap and exercise

You built an agent from first principles — the describe/call/execute/feed-back
loop — and saw the same thing through a framework.

**Exercise.** Add a third tool, `get_time()`, that returns the
current time, and ask the agent a question that needs two of the three tools.
Confirm it chains the calls correctly.